# Tool Calling example

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_openai import ChatOpenAI

In [3]:
llm = ChatOpenAI(model="gpt-4o-mini")

In [4]:
output = llm.invoke("What's the lake near the airport at the capital of Uganda?")

In [5]:
print(output.content)

The lake near the airport in Kampala, Uganda, is Lake Victoria. Entebbe International Airport, which serves the capital, is situated near the shores of this large lake, which is one of the largest freshwater lakes in the world.


In [6]:
output.tool_calls

[]

In [7]:
import random

In [8]:
from langchain_core.tools import tool

In [9]:
@tool
def get_fake_weather(city: str) -> str:
    """
    Args:
        city: The name of the city for which the weather is expected
    Output:
        Weather summary as a string
    """
    return f"{random.choice(['Sunny', 'Cloudy', 'Rainy', 'Windy'])}, {random.randint(10,30)}°C"

In [10]:
@tool
def convert_euro_to_dollar(amount: float) -> float:
    """Convert the currency amount from Euro to Dollar
    Args:
        amount: the given amount in in Euros, €
    Returns:
        amount in dollar, $
    """
    return amount/0.87


@tool
def convert_dollar_to_euro(amount: float) -> float:
    """Convert the currency amount from Dollar to Euro
    Args:
        amount: the given amount in dollar, $
    Returns:
        amount in Euros, €
    """
    return 0.87*amount

In [11]:
tools = [get_fake_weather, convert_euro_to_dollar, convert_dollar_to_euro]

In [15]:
get_fake_weather.invoke({"city": "Cologne"})

'Rainy, 19°C'

In [16]:
convert_dollar_to_euro.invoke({"amount": 100})

87.0

In [17]:
convert_euro_to_dollar.invoke({"amount": 87})

100.0

In [18]:
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage

In [19]:
messages = [
    SystemMessage("You are a helpful assistant that helps plan tours in cities around the world. You also have access to tools when available, so use them when needed! Consider localization of weather, currency and other factors while advising. Be precise and short"),
    HumanMessage("What is the weather in Cologne, Germany, tomorrow?")
]

In [20]:
llm.invoke(messages)

AIMessage(content="I currently don't have access to real-time data, including weather updates. I recommend checking a reliable weather website or app for the most accurate and up-to-date forecast for Cologne, Germany, tomorrow.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 39, 'prompt_tokens': 68, 'total_tokens': 107, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_28f3a9c586', 'id': 'chatcmpl-ETpPEHha01bLo4qTQ5YkjuvX4LKs5', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0f2ad-1330-7603-bd3c-80c1ed904dff-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={

In [21]:
llm_with_tools = llm.bind_tools(tools)

In [22]:
output_message = llm_with_tools.invoke(messages)
output_message

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 216, 'total_tokens': 232, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_c77bee8c7a', 'id': 'chatcmpl-ETpPGV3AqMG2X6f0rAgkVmrsL6G6U', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0f2ad-1a2b-7d11-a5b9-0956af18b59a-0', tool_calls=[{'name': 'get_fake_weather', 'args': {'city': 'Cologne'}, 'id': 'call_lWgIr1iDg4EFTOxPVUirzDqz', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 216, 'output_tokens': 16, 'total_tokens': 232, 'input_token_details'

In [23]:
output_message.tool_calls

[{'name': 'get_fake_weather',
  'args': {'city': 'Cologne'},
  'id': 'call_lWgIr1iDg4EFTOxPVUirzDqz',
  'type': 'tool_call'}]

In [24]:
messages

[SystemMessage(content='You are a helpful assistant that helps plan tours in cities around the world. You also have access to tools when available, so use them when needed! Consider localization of weather, currency and other factors while advising. Be precise and short', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='What is the weather in Cologne, Germany, tomorrow?', additional_kwargs={}, response_metadata={})]

In [25]:
messages.append(output_message)

In [26]:
messages

[SystemMessage(content='You are a helpful assistant that helps plan tours in cities around the world. You also have access to tools when available, so use them when needed! Consider localization of weather, currency and other factors while advising. Be precise and short', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='What is the weather in Cologne, Germany, tomorrow?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 216, 'total_tokens': 232, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_c77bee

In [27]:
tool_map = {"get_fake_weather": get_fake_weather, "convert_euro_to_dollar": convert_euro_to_dollar, "convert_dollar_to_euro": convert_dollar_to_euro}

In [28]:
for tool_used in output_message.tool_calls:
    actual_tool_object = tool_map.get(tool_used.get("name"))
    if actual_tool_object:
        args = tool_used.get("args")
        tool_id = tool_used.get("id")
        tool_output = actual_tool_object.invoke(args)
        tool_output_message = ToolMessage(content=tool_output, tool_call_id=tool_id)
        messages.append(tool_output_message)

In [29]:
messages

[SystemMessage(content='You are a helpful assistant that helps plan tours in cities around the world. You also have access to tools when available, so use them when needed! Consider localization of weather, currency and other factors while advising. Be precise and short', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='What is the weather in Cologne, Germany, tomorrow?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 216, 'total_tokens': 232, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_c77bee

In [30]:
weather_output = llm_with_tools.invoke(messages)
weather_output

AIMessage(content="Tomorrow's weather in Cologne, Germany, is expected to be cloudy with a temperature of 26°C.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 247, 'total_tokens': 269, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_c77bee8c7a', 'id': 'chatcmpl-ETpPHCgsfkP22egW3m0VXWl3q0fLl', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0f2ad-1f35-7932-8345-9726f45aa565-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 247, 'output_tokens': 22, 'total_tokens': 269, 'input_token_details': {'audio': 0, 'cache_read': 0}

In [31]:
messages.append(weather_output)

In [32]:
messages

[SystemMessage(content='You are a helpful assistant that helps plan tours in cities around the world. You also have access to tools when available, so use them when needed! Consider localization of weather, currency and other factors while advising. Be precise and short', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='What is the weather in Cologne, Germany, tomorrow?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 216, 'total_tokens': 232, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_c77bee

In [33]:
messages.append(
    HumanMessage("""
        I have $120 travelling from US to spend in this new city. My transactions/expenditure in the city are all in the local currency. 
        What will be left at the end of the day if I give 50 to my friend there and spend 20 for public transport and another 30 for food?
        """)
    )

In [34]:
messages

[SystemMessage(content='You are a helpful assistant that helps plan tours in cities around the world. You also have access to tools when available, so use them when needed! Consider localization of weather, currency and other factors while advising. Be precise and short', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='What is the weather in Cologne, Germany, tomorrow?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 216, 'total_tokens': 232, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_c77bee

In [35]:
cologne_tour_output = llm_with_tools.invoke(messages)
cologne_tour_output

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 70, 'prompt_tokens': 342, 'total_tokens': 412, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_c77bee8c7a', 'id': 'chatcmpl-ETpPJh2NmC1riuWKlFb1lwPH9BZHH', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0f2ad-2478-7650-8025-7be9b489e3be-0', tool_calls=[{'name': 'convert_dollar_to_euro', 'args': {'amount': 50}, 'id': 'call_8KZblsx9MSI5uRdI8hhnyxmO', 'type': 'tool_call'}, {'name': 'convert_dollar_to_euro', 'args': {'amount': 20}, 'id': 'call_HclMDfG0I7PBVzXpDsAftTQY', 'type': 'tool_call'}, {'n

In [36]:
cologne_tour_output.tool_calls

[{'name': 'convert_dollar_to_euro',
  'args': {'amount': 50},
  'id': 'call_8KZblsx9MSI5uRdI8hhnyxmO',
  'type': 'tool_call'},
 {'name': 'convert_dollar_to_euro',
  'args': {'amount': 20},
  'id': 'call_HclMDfG0I7PBVzXpDsAftTQY',
  'type': 'tool_call'},
 {'name': 'convert_dollar_to_euro',
  'args': {'amount': 30},
  'id': 'call_Vc9Z9yCAfSq7tXgPgMY4zoyr',
  'type': 'tool_call'}]

In [37]:
messages.append(cologne_tour_output)

In [38]:
for tool_call_obj in cologne_tour_output.tool_calls:
    tool_object = tool_map.get(tool_call_obj.get("name"))
    if tool_object:
        args = tool_call_obj.get("args")
        tool_id = tool_call_obj.get("id")
        tool_output = tool_object.invoke(args)
        tool_msg = ToolMessage(tool_output, tool_call_id=tool_id)
        messages.append(tool_msg)

In [39]:
tool_object

StructuredTool(name='convert_dollar_to_euro', description='Convert the currency amount from Dollar to Euro\n    Args:\n        amount: the given amount in dollar, $\n    Returns:\n        amount in Euros, €', args_schema=<class 'langchain_core.utils.pydantic.convert_dollar_to_euro'>, func=<function convert_dollar_to_euro at 0x0000029CFEB37D80>)

In [40]:
messages

[SystemMessage(content='You are a helpful assistant that helps plan tours in cities around the world. You also have access to tools when available, so use them when needed! Consider localization of weather, currency and other factors while advising. Be precise and short', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='What is the weather in Cologne, Germany, tomorrow?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 216, 'total_tokens': 232, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_c77bee

In [41]:
final_output = llm_with_tools.invoke(messages)

In [42]:
print(final_output.content)

Here are the conversions for your expenses in Cologne:

- Giving $50 to your friend: €43.50
- Spending $20 for public transport: €17.40
- Spending $30 for food: €26.10

Total expenditure in Euros: €43.50 + €17.40 + €26.10 = €87.00

Starting with $120, if you convert it to Euros:

$120 is approximately €105.60.

Amount left after spending: €105.60 - €87.00 = €18.60

You will have approximately $22.00 left at the end of the day.


In [44]:
messages.extend([final_output, HumanMessage("And how much can I bring back to the US in my original curency?")])

In [45]:
messages

[SystemMessage(content='You are a helpful assistant that helps plan tours in cities around the world. You also have access to tools when available, so use them when needed! Consider localization of weather, currency and other factors while advising. Be precise and short', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='What is the weather in Cologne, Germany, tomorrow?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 216, 'total_tokens': 232, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_c77bee

In [46]:
output_after_trip = llm_with_tools.invoke(messages)

In [47]:
messages.append(output_after_trip)

In [48]:
output_after_trip.tool_calls

[{'name': 'convert_euro_to_dollar',
  'args': {'amount': 18.6},
  'id': 'call_NGVMogjPMTh1oyhRXjBQafan',
  'type': 'tool_call'}]

In [49]:
for tool_call in output_after_trip.tool_calls:
    tool_call_obj = tool_map.get(tool_call.get("name"))
    if tool_call_obj:
        args = tool_call.get("args")
        tool_id = tool_call.get("id")
        tool_msg = tool_call_obj.invoke(args)
        messages.append(ToolMessage(tool_msg, tool_call_id=tool_id))

In [50]:
messages

[SystemMessage(content='You are a helpful assistant that helps plan tours in cities around the world. You also have access to tools when available, so use them when needed! Consider localization of weather, currency and other factors while advising. Be precise and short', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='What is the weather in Cologne, Germany, tomorrow?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 216, 'total_tokens': 232, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_c77bee

In [51]:
llm_with_tools.invoke(messages)

AIMessage(content='You can bring back approximately $21.38 to the US in your original currency.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 630, 'total_tokens': 648, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_c77bee8c7a', 'id': 'chatcmpl-ETpSlmlEZHz0RiR2J3vDpGDrpqFzv', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0f2b0-6879-7b52-9585-d9dcd3bb43fe-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 630, 'output_tokens': 18, 'total_tokens': 648, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_

## Steps for Tool calling
1. Imports
    - `langchain_openai.ChatOpenAI`
    - `langchain_core.tools.tool`
    - `langchain_core.messages.HumanMessage`
    - `langchain_core.messages.ToolMessage`
2. Create a chat object `llm`, that can be called with the `.invoke` method.
3. Write `@tool` decorated function/s, with arguments and output along with documentation
4. Create a list of these tools, called `tools`
5. Create a new llm object `llm_with_tools`, and bind the `tools` list with it
    - `llm_with_tools = llm.bind_tools(tools)`
6. Create a list of `messages`, initialized with the `HumanMessage` with the specific question
7. Call the `llm_with_tools.invoke` method with the `messages` as a history, and append the output message, `output_message` into the `messages`
    - The invocation does not call the tool!!!
8. Create a `tool_map` that has keys as the string names of the functions, and the function objects as the values.
9. Loop through the `tool_calls` in the `output_message` and extract for each tool, the right function tool using the `tool_map`
    - Extract the arguments: `tool_used.get("args")`
    - Extract the `tool_id = tool_used.get("id")`
    - Use the `args`, along with the `actual_tool_object` by using it's `invoke` method, to get the right output, as if calling the function directly
    - Create a `ToolMessage` output using the above output along with the `tool_id`, and append to the list of `messages`
10. Call the `invoke` method of the `llm_with_tools` one final time, with the entire `messages` list as chat history, to get the final output.
